In [ ]:
# Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q open-clip-torch sentence-transformers langchain-text-splitters tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00


In [ ]:
import json
import os
import torch
import numpy as np
import open_clip
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Cấu hình thiết bị
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("⏳ Đang tải Model CLIP (cho Summary)...")
model_clip, _, _ = open_clip.create_model_and_transforms('ViT-H-14', pretrained='laion2b_s32b_b79k')
tokenizer_clip = open_clip.get_tokenizer('ViT-H-14')
model_clip = model_clip.to(device).eval()

print("⏳ Đang tải Model E5 (cho Chunks)...")
model_text = SentenceTransformer('intfloat/multilingual-e5-large', device=device)

# Số chiều vector (Cần báo cho bạn mình biết để tạo Collection trong Milvus)
DIM_CLIP = 1024
DIM_E5 = 1024


⏳ Đang tải Model CLIP (cho Summary)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


⏳ Đang tải Model E5 (cho Chunks)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import re
import json
import torch
import numpy as np
from tqdm.notebook import tqdm

# --- 1. HÀM KIỂM TRA VECTOR (GIỮ NGUYÊN NHƯNG THÊM CHECK NAN) ---
def is_valid_vec(vec, expected_dim):
    if vec is None: return False
    v = np.array(vec)
    if v.size == 0 or v.shape[0] != expected_dim: return False
    if np.isnan(v).any() or np.isinf(v).any(): return False
    return True

# --- 2. HÀM DỌN RÁC WIKIPEDIA ---
def clean_wiki_text(text):
    if not text: return ""
    # Xóa tham chiếu [1], [2], [citation needed]
    text = re.sub(r'\[\d+\]|\[citation needed\]', '', text)
    # Xóa khoảng trắng thừa, xuống dòng dư thừa
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# GIỮ NGUYÊN TÊN NHƯNG GÁN THÊM ĐỂ CODE CHÍNH KHÔNG LỖI
clean_txt = clean_wiki_text

# --- 3. HÀM CHUNK LOGIC (MỚI: GIỮ HEADER ĐỂ GIÀU NGỮ CẢNH) ---
def enrich_and_split(item, splitter):
    title = item.get("title", "Unknown")
    full_text = item.get("full_text", "")

    if not full_text or len(full_text.strip()) < 20:
        return []

    # Tách văn bản dựa trên các Header == Geography ==, == History ==
    # Giữ lại Header để làm context cho từng chunk
    parts = re.split(r'(==+.*?==+)', full_text)

    current_header = "General Information"
    final_chunks = []

    for part in parts:
        part = part.strip()
        if not part: continue

        # Nếu là Header thì cập nhật Header hiện tại
        if re.match(r'==+.*?==+', part):
            current_header = part.strip('= ').strip()
            continue

        # Bỏ qua các mục rác cuối bài Wikipedia
        if current_header.lower() in ['references', 'see also', 'external links', 'notes']:
            continue

        # Dọn dẹp nội dung dưới Header
        clean_part = clean_wiki_text(part)
        if not clean_part: continue

        # Chia nhỏ nội dung
        sub_chunks = splitter.split_text(clean_part)

        for c in sub_chunks:
            # Gộp Title + Header vào nội dung để Vector hóa "giàu" hơn
            # Kết quả: "[Ba Vì - Geography] Ba Vì is a rural district..."
            enriched_content = f"[{title} - {current_header}] {c}"
            final_chunks.append(enriched_content)

    return final_chunks

In [ ]:
import os
import json
import traceback

# --- CẤU HÌNH ĐƯỜNG DẪN ---
INPUT_JSON = '/content/drive/MyDrive/merged_data_final.json'

# 1. Khai báo thư mục chứa toàn bộ Output
OUTPUT_DIR = '/content/drive/MyDrive/Milvus_Export_Data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Đưa tất cả file kết quả và file log vào thư mục Output
OUT_SUMMARY_JSONL = os.path.join(OUTPUT_DIR, 'milvus_summary.jsonl')
OUT_CHUNK_JSONL   = os.path.join(OUTPUT_DIR, 'milvus_chunks.jsonl')
PROGRESS_FILE     = os.path.join(OUTPUT_DIR, 'processed_ids.txt')
ERROR_LOG_FILE    = os.path.join(OUTPUT_DIR, 'error_log.txt')

def load_processed_ids():
    """Đọc danh sách các ID đã xử lý từ trước"""
    processed_ids = set()
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                processed_ids.add(line.strip())
    return processed_ids

def process_and_export():
    splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)

    # 1. Load dữ liệu gốc
    with open(INPUT_JSON, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 2. Load các ID đã xử lý để bỏ qua
    processed_ids = load_processed_ids()
    print(f"🔄 Đã tìm thấy {len(processed_ids)} địa điểm đã xử lý. Sẽ tự động bỏ qua...")

    # KHỞI TẠO BIẾN ĐẾM
    count_summary = 0
    count_chunk = 0

    # 3. Mở các file ở chế độ 'a' (append)
    with open(OUT_SUMMARY_JSONL, 'a', encoding='utf-8') as f_sum, \
         open(OUT_CHUNK_JSONL, 'a', encoding='utf-8') as f_chunk, \
         open(PROGRESS_FILE, 'a', encoding='utf-8') as f_prog, \
         open(ERROR_LOG_FILE, 'a', encoding='utf-8') as f_err:

        # Gán tqdm vào một biến pbar để có thể cập nhật log realtime
        pbar = tqdm(data, desc="Đang xử lý 130k địa danh")

        for item in pbar:
            idx = str(item.get("id_dia_diem"))
            title = item.get("title", "Unknown")

            if idx in processed_ids:
                continue

            try:
                # --- 1. Xử lý SUMMARY (CLIP) ---
                s_txt = f"[{title}] {item.get('summary_text', '')}"
                with torch.no_grad():
                    tokens = tokenizer_clip([clean_txt(s_txt)]).to(device)
                    s_vec = model_clip.encode_text(tokens)
                    s_vec /= s_vec.norm(dim=-1, keepdim=True)
                    s_vec = s_vec.cpu().numpy()[0]

                if is_valid_vec(s_vec, DIM_CLIP):
                    f_sum.write(json.dumps({
                        "id": idx, "vector": s_vec.tolist(), "title": title
                    }, ensure_ascii=False) + "\n")

                    # Tăng biến đếm summary
                    count_summary += 1

                # --- 2. Xử lý CHUNKS (E5) ---
                chunks = enrich_and_split(item, splitter)
                if chunks:
                    texts_for_e5 = ["passage: " + clean_txt(c) for c in chunks]
                    c_vecs = model_text.encode(texts_for_e5, batch_size=32, show_progress_bar=False)

                    for j, (txt, v) in enumerate(zip(chunks, c_vecs)):
                        if is_valid_vec(v, DIM_E5):
                            f_chunk.write(json.dumps({
                                "chunk_id": f"{idx}_{j}",
                                "parent_id": idx,
                                "vector": v.tolist(),
                                "content": txt
                            }, ensure_ascii=False) + "\n")

                            # Tăng biến đếm chunk
                            count_chunk += 1

                # --- 3. ĐÁNH DẤU LÀ ĐÃ XONG ID NÀY ---
                f_prog.write(idx + "\n")

                # ÉP GHI DỮ LIỆU XUỐNG DRIVE NGAY LẬP TỨC
                f_sum.flush()
                f_chunk.flush()
                f_prog.flush()

                # CẬP NHẬT SỐ LƯỢNG LÊN THANH TIẾN TRÌNH REALTIME
                pbar.set_postfix({"Sum": count_summary, "Chunk": count_chunk})

            except Exception as e:
                # --- 4. GHI LOG LỖI VÀ CHẠY TIẾP ---
                error_msg = f"Lỗi ở ID {idx} - Title: {title}\nChi tiết lỗi: {str(e)}\n{traceback.format_exc()}\n{'-'*50}\n"
                f_err.write(error_msg)
                f_err.flush()
                continue

    # IN LOG TỔNG KẾT KHI CHẠY XONG
    print("\n" + "="*50)
    print(f"🏁 HOÀN TẤT QUÁ TRÌNH TẠO VECTOR!")
    print(f"✅ Tổng số Summary vector đã tạo: {count_summary}")
    print(f"✅ Tổng số Chunk vector đã tạo: {count_chunk}")
    print(f"📁 Dữ liệu được lưu tại: {OUTPUT_DIR}")
    print("="*50)

# Chạy quy trình
process_and_export()

🔄 Đã tìm thấy 4147 địa điểm đã xử lý. Sẽ tự động bỏ qua...


Đang xử lý 130k địa danh:   0%|          | 0/130708 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# # Cấu hình đường dẫn
# INPUT_JSON = '/content/drive/MyDrive/vietnam_only.json'
# OUT_SUMMARY_JSONL = '/content/drive/MyDrive/milvus_summaryVNNN.jsonl'
# OUT_CHUNK_JSONL = '/content/drive/MyDrive/milvus_chunksVNNN.jsonl'

# def process_and_export():
#     splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)

#     with open(INPUT_JSON, 'r', encoding='utf-8') as f:
#         data = json.load(f)

#     # Dùng mode 'a' để có thể chạy tiếp nếu Colab bị ngắt quãng
#     with open(OUT_SUMMARY_JSONL, 'a', encoding='utf-8') as f_sum, \
#          open(OUT_CHUNK_JSONL, 'a', encoding='utf-8') as f_chunk:

#         for item in tqdm(data, desc="Đang xử lý 130k địa danh"):
#             idx = item.get("id_dia_diem")
#             title = item.get("title", "Unknown")

#             try:
#                 # 1. Xử lý SUMMARY (CLIP)
#                 s_txt = f"[{title}] {item.get('summary_text', '')}"
#                 with torch.no_grad():
#                     tokens = tokenizer_clip([clean_txt(s_txt)]).to(device)
#                     s_vec = model_clip.encode_text(tokens)
#                     s_vec /= s_vec.norm(dim=-1, keepdim=True)
#                     s_vec = s_vec.cpu().numpy()[0]

#                 if is_valid_vec(s_vec, DIM_CLIP):
#                     f_sum.write(json.dumps({
#                         "id": idx, "vector": s_vec.tolist(), "title": title
#                     }, ensure_ascii=False) + "\n")

#                 # 2. Xử lý CHUNKS (E5)
#                 chunks = enrich_and_split(item, splitter)
#                 if chunks:
#                     # Model E5 cần tiền tố 'passage: ' để đạt hiệu quả cao nhất
#                     texts_for_e5 = ["passage: " + clean_txt(c) for c in chunks]
#                     c_vecs = model_text.encode(texts_for_e5, batch_size=32, show_progress_bar=False)

#                     for j, (txt, v) in enumerate(zip(chunks, c_vecs)):
#                         if is_valid_vec(v, DIM_E5):
#                             f_chunk.write(json.dumps({
#                                 "chunk_id": f"{idx}_{j}",
#                                 "parent_id": idx,
#                                 "vector": v.tolist(),
#                                 "content": txt
#                             }, ensure_ascii=False) + "\n")

#             except Exception as e:
#                 # Ghi log lỗi để không làm dừng toàn bộ chương trình
#                 continue

#     print("🏁 Hoàn tất! Bạn có thể lấy 2 file .jsonl trên Drive đưa cho bạn mình.")

# # Chạy quy trình
# process_and_export()

In [ ]:
# import json
# import torch
# from sentence_transformers import util

# # Đường dẫn file bạn vừa export
# PATH_SUMMARY = '/content/drive/MyDrive/milvus_summaryVNNN.jsonl'
# PATH_CHUNKS = '/content/drive/MyDrive/milvus_chunksVNNN.jsonl'

# summary_data = []
# chunk_data = []

# print("📖 Loading Summary Data...")
# with open(PATH_SUMMARY, 'r', encoding='utf-8') as f:
#     for line in f: summary_data.append(json.loads(line))

# print("📖 Loading Chunk Data...")
# with open(PATH_CHUNKS, 'r', encoding='utf-8') as f:
#     for line in f: chunk_data.append(json.loads(line))

# # Chuyển sang Tensor để GPU xử lý siêu nhanh
# summary_vecs = torch.tensor([r['vector'] for r in summary_data]).to(device)
# chunk_vecs = torch.tensor([r['vector'] for r in chunk_data]).to(device)

# print(f"✅ Ready! {len(summary_data)} Locations and {len(chunk_data)} Chunks loaded.")

In [ ]:
# def final_test_search(query_en, top_k_id=10, top_k_chunks=10):
#     print(f"🔎 Query: '{query_en}'")
#     print("="*70)

#     # --- BƯỚC 1: Dùng CLIP tìm ID (Dựa trên Summary) ---
#     with torch.no_grad():
#         tokens = tokenizer_clip([query_en[:70]]).to(device) # Giới hạn token cho CLIP
#         query_clip_vec = model_clip.encode_text(tokens)
#         query_clip_vec /= query_clip_vec.norm(dim=-1, keepdim=True)

#     # Tính tương đồng Summary
#     s_scores = util.cos_sim(query_clip_vec, summary_vecs)[0]
#     top_id_indices = torch.topk(s_scores, k=top_k_id).indices
#     potential_ids = [summary_data[i]['id'] for i in top_id_indices]

#     print(f"📍 Step 1 (ViT) identified IDs: {potential_ids}")

#     # --- BƯỚC 2: Dùng E5 search sâu vào Chunks của các ID này ---
#     # Lọc nhanh các chunks thuộc về ID đã chọn
#     valid_chunks = []
#     valid_vecs = []
#     for i, c in enumerate(chunk_data):
#         if c['parent_id'] in potential_ids:
#             valid_chunks.append(c)
#             valid_vecs.append(chunk_vecs[i])

#     if not valid_vecs:
#         print("❌ No chunks found for these IDs.")
#         return

#     # Encode query bằng E5
#     with torch.no_grad():
#         query_e5_vec = model_text.encode(f"query: {query_en}", convert_to_tensor=True).to(device)

#     # Tính tương đồng trên tập đã lọc
#     v_stack = torch.stack(valid_vecs).to(device)
#     c_scores = util.cos_sim(query_e5_vec, v_stack)[0]
#     top_c_results = torch.topk(c_scores, k=min(top_k_chunks, len(c_scores)))

#     print(f"📝 Step 2 (E5) Top Relevant Chunks:")
#     for score, idx in zip(top_c_results.values, top_c_results.indices):
#         res = valid_chunks[idx]
#         print(f"⭐ [Score: {score:.4f}] [ID: {res['parent_id']}]")
#         print(f"Content: {res['content']}") # Sẽ thấy rõ Header [Title - Geography]
#         print("-" * 40)

# # --- THỬ NGHIỆM THỰC TẾ ---
# final_test_search("Đống Đa is one of the four original urban districts (quận) of Hanoi")

In [ ]:
# eval_set_easy = [
#     # Ba Vì
#     {"query": "Information about Ba Vi District and National Park", "expected_id": 164516},
#     {"query": "Ba Vi mountain range in Vietnam", "expected_id": 164516},
#     # Bến Cát
#     {"query": "Ben Cat city in Binh Duong province", "expected_id": 164515},
#     {"query": "A city located 50 km north of Ho Chi Minh City", "expected_id": 164515},
#     # TP.HCM
#     {"query": "General information about Ho Chi Minh City", "expected_id": 164170},
#     {"query": "Financial and economic center of Vietnam", "expected_id": 164170},
#     {"query": "Saigon city history and population", "expected_id": 164170},
#     # Ba Tơ
#     {"query": "Ba To rural district in Quang Ngai", "expected_id": 164518},
#     {"query": "Geography and climate of Ba To district", "expected_id": 164518},
#     {"query": "Unknown fatal disease in Ba To area", "expected_id": 164518},
#     # Các câu hỏi trực tiếp khác
#     {"query": "Total natural area of Ben Cat city", "expected_id": 164515},
#     {"query": "Population of Ho Chi Minh City in 2019", "expected_id": 164170},
#     {"query": "District covers an area of 1,133 km2 in Quang Ngai", "expected_id": 164518},
#     {"query": "Official name of Saigon after 1956", "expected_id": 164170},
#     {"query": "Is Ben Cat a grade 3 urban area?", "expected_id": 164515}
# ]

In [ ]:
# def run_advanced_evaluation(test_set, top_k_vit=10):
#     total = len(test_set)
#     hits = 0
#     mrr_score = 0

#     print(f"📊 Đang chấm điểm Retrieval (Top {top_k_vit} ViT)...")
#     print("-" * 50)

#     for item in test_set:
#         query = item['query']
#         expected_id = item['expected_id']

#         # --- BƯỚC 1: Tìm kiếm bằng ViT (CLIP) ---
#         with torch.no_grad():
#             tokens = tokenizer_clip([query[:70]]).to(device)
#             q_vec = model_clip.encode_text(tokens)
#             q_vec /= q_vec.norm(dim=-1, keepdim=True)

#         # Tính tương đồng với Summary
#         scores = util.cos_sim(q_vec, summary_vecs)[0]
#         top_indices = torch.topk(scores, k=top_k_vit).indices
#         retrieved_ids = [summary_data[i]['id'] for i in top_indices]

#         # --- KIỂM TRA KẾT QUẢ ---
#         if expected_id in retrieved_ids:
#             hits += 1
#             # Tính rank (vị trí)
#             rank = retrieved_ids.index(expected_id) + 1
#             mrr_score += (1.0 / rank)
#             print(f"✅ Pass: '{query[:40]}...' -> Rank: {rank}")
#         else:
#             print(f"❌ Fail: '{query[:40]}...'")

#     # Tính toán con số cuối cùng
#     final_hit_rate = hits / total
#     final_mrr = mrr_score / total

#     print("-" * 50)
#     print(f"📈 KẾT QUẢ CUỐI CÙNG:")
#     print(f"🎯 Hit Rate@{top_k_vit}: {final_hit_rate:.2%}")
#     print(f"🚀 MRR: {final_mrr:.4f}")
#     print("-" * 50)

# # Chạy đánh giá với bộ câu hỏi dễ
# run_advanced_evaluation(eval_set_easy, top_k_vit=10)

📊 Đang chấm điểm Retrieval (Top 10 ViT)...
--------------------------------------------------
✅ Pass: 'Information about Ba Vi District and Nat...' -> Rank: 1
✅ Pass: 'Ba Vi mountain range in Vietnam...' -> Rank: 1
✅ Pass: 'Ben Cat city in Binh Duong province...' -> Rank: 1
❌ Fail: 'A city located 50 km north of Ho Chi Min...'
✅ Pass: 'General information about Ho Chi Minh Ci...' -> Rank: 1
✅ Pass: 'Financial and economic center of Vietnam...' -> Rank: 3
✅ Pass: 'Saigon city history and population...' -> Rank: 1
✅ Pass: 'Ba To rural district in Quang Ngai...' -> Rank: 5
❌ Fail: 'Geography and climate of Ba To district...'
✅ Pass: 'Unknown fatal disease in Ba To area...' -> Rank: 1
✅ Pass: 'Total natural area of Ben Cat city...' -> Rank: 1
✅ Pass: 'Population of Ho Chi Minh City in 2019...' -> Rank: 1
❌ Fail: 'District covers an area of 1,133 km2 in ...'
✅ Pass: 'Official name of Saigon after 1956...' -> Rank: 5
✅ Pass: 'Is Ben Cat a grade 3 urban area?...' -> Rank: 1
------------------